# Lab 13 challenge: worked solution

For the instructor. Both tasks use only what Lab 13 showed. The first two cells are the lab's; the third imports the
lab's functions from `space/app.py`, the file Part F checked against the notebook.

In [1]:
# Setup: on Colab, copy the repository and install the packages. Locally (after uv sync) nothing is installed.
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("/content/agentic_ai_conversational_agents").exists():
        !git clone -q --depth 1 https://github.com/tatwan/agentic_ai_conversational_agents.git /content/agentic_ai_conversational_agents
    %cd /content/agentic_ai_conversational_agents
    %pip install -q -r requirements.txt

ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents] if (folder / "models.env").exists())
sys.path.insert(0, str(ROOT))
print("Repository folder:", ROOT.name)

Repository folder: agentic_ai_conversational_agents


In [2]:
# Config: model names come from models.env; the key from the environment, .env or Colab Secrets.
from shared.settings import load_settings, load_key

settings = load_settings()
key_found_in = load_key("OPENAI_API_KEY")

for name in ("CHAT_MODEL", "REASONING_EFFORT", "FALLBACK_MODEL", "EMBED_MODEL", "MODERATION_MODEL"):
    print(f"{name:18}{settings[name] or '(not set)'}")
print(f"{'OPENAI_API_KEY':18}{'found in ' + key_found_in if key_found_in else 'not found'}")

CHAT_MODEL        gpt-4o-mini
REASONING_EFFORT  (not set)
FALLBACK_MODEL    gpt-4o-mini
EMBED_MODEL       text-embedding-3-small
MODERATION_MODEL  omni-moderation-latest
OPENAI_API_KEY    found in .env


In [3]:
import warnings
warnings.filterwarnings("ignore", message="IProgress not found")
import gradio as gr
import httpx
from gradio_client import Client
from langchain.agents import create_agent
from langchain_core.messages import AIMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

sys.path.insert(0, str(ROOT / "13_Deploy_To_A_Space" / "space"))
import app as page_app                    # Lab 13's seven functions, unchanged
from app import SAFE_MESSAGE, SAFE_REPLY, STILL_WAITING, approval_message, checked, next_turn, tool_step
from shared.guards import PLANTED
from shared.ops_api import start_ops_api
from shared.ops_tools import API, RepContext
from shared.scripted import scripted

page_app.settings = settings              # the app reads models.env from its own folder; use the repository's
server = start_ops_api()
LAUNCH = dict(prevent_thread_lock=True, share=False, inline=IN_COLAB, quiet=True)
REP = RepContext(rep_id="rep-3", rep_name="Jane Peacock")
TICKET_14 = ("Ticket 14: a customer was charged twice for the same album, invoices 9001 and 9002. "
             "Please file the refund.")
FAIR = "Summarise the newest note on the duplicate-charge case for me."

def refund_requests():
    found, n = [], 1
    while (response := httpx.get(f"{API}/refund-requests/{n}")).status_code == 200:
        found.append(response.json())
        n += 1
    return found

Ops API started at http://127.0.0.1:8765


## Task 1: Keep a waiting refund across a reload

The only change to the lab's `respond` is the thread id: `request.username`, which the sign-in fills in, instead of
`request.session_hash`.

In [4]:
def respond_signed_in(message, history, request: gr.Request):
    cfg = {"configurable": {"thread_id": request.username}}      # was: request.session_hash
    waiting = copilot.get_state(cfg).interrupts
    turn = next_turn(message, waiting)
    if turn is None:                # a write waits and this is no decision: remind the rep, leave the thread alone
        yield [gr.ChatMessage(content=STILL_WAITING), approval_message(waiting)]
        return
    shown = []
    try:
        for item in copilot.stream(turn, cfg, context=REP, stream_mode="messages", version="v2"):
            token, meta = item["data"]
            if token.type == "tool":
                shown.append(tool_step(token))
                yield shown
    except Exception:
        yield shown + [gr.ChatMessage(content=SAFE_MESSAGE)]
        return
    state = copilot.get_state(cfg)
    if state.interrupts:
        yield shown + [approval_message(state.interrupts)]
        return
    reply, failed = checked(state.values["messages"][-1].text)
    yield shown + [gr.ChatMessage(content=reply)]

In [5]:
copilot = page_app.build_copilot()
USERS = [("jane", "jane-password"), ("steve", "steve-password")]
page = gr.ChatInterface(respond_signed_in)
page.launch(**LAUNCH, auth=USERS)

jane = Client(page.local_url, auth=USERS[0], verbose=False)
print("jane:", jane.predict(TICKET_14, api_name="/respond_signed_in")[-1]["content"][:110])
steve = Client(page.local_url, auth=USERS[1], verbose=False)
print("steve asks what is waiting:",
      steve.predict("Is anything waiting for my approval?", api_name="/respond_signed_in")[-1]["content"][:200])

jane_reloaded = Client(page.local_url, auth=USERS[0], verbose=False)
before = len(refund_requests())
print("jane after the reload:", jane_reloaded.predict("approve", api_name="/respond_signed_in")[-1]["content"][:200])
print("filed:", [(r["invoice_id"], r["amount"], r["status"]) for r in refund_requests()[before:]])
page.close()

jane: Waiting for your approval: file_refund_request {'invoice_id': 9002, 'amount': 9.99, 'reason': 'duplicate_charg


steve asks what is waiting: The handbook does not cover pending approvals, so please ask the support manager for clarification.


jane after the reload: I have filed a refund request for invoice 9002 due to a duplicate charge, and it is pending review [policy_refunds.md]. 

Suggested reply: "I've submitted a request to refund the duplicate charge. You
filed: [(9002, 9.99, 'pending_review')]


Closing server running on port: 7860


**Result.** Jane's refund for invoice 9002 waited; after the reload, a new client signed in as `jane` sent *approve*, and the request was filed as `pending_review`, where the lab's reload filed nothing. Steve's conversation knew nothing of Jane's: asked what was waiting for his approval, the copilot said the handbook does not cover pending approvals.

What you lose: every tab and every device Jane opens shares one thread, so two cases at once mix in one conversation.
A real app keys the thread on the rep and the case (`f"{username}:{case_id}"`) and keeps threads in a database, so a
restart does not lose them either.

The replies suggest the threads are apart; the checkpointer shows it. Read each visitor's thread: its messages, the
first one, whether any names invoice 9002, and whether a write still waits.

In [6]:
for name in ("jane", "steve"):
    state = copilot.get_state({"configurable": {"thread_id": name}})
    messages = state.values["messages"]
    print(f"{name:6}thread: {len(messages)} messages, first {messages[0].text[:50]!r} | "
          f"names invoice 9002: {any('9002' in m.text for m in messages)} | write waiting: {bool(state.interrupts)}")

jane  thread: 8 messages, first 'Ticket 14: a customer was charged twice for the sa' | names invoice 9002: True | write waiting: False
steve thread: 2 messages, first 'Is anything waiting for my approval?' | names invoice 9002: False | write waiting: False


Jane's thread holds her 8 messages, from ticket 14 to the filed refund, names invoice 9002 and has nothing waiting;
Steve's holds his 2 messages, his question and the reply, and never names 9002. A reply can be vague for many reasons;
the saved threads show the conversations are apart.

## Task 2 (optional): Check the reply a sentence at a time

The page collects the model's words in `pending`, splits off every finished sentence (one that ends with `.`, `?` or
`!` and a space), checks it, and shows it only if it passes. The first failure ends the turn with `SAFE_REPLY`. The
finished reply is checked once more at the end, for the last sentence.

In [7]:
import re

SENTENCE_END = re.compile(r"(?<=[.!?])\s+")

def respond_sentences(message, history, request: gr.Request):
    cfg = {"configurable": {"thread_id": request.session_hash}}
    shown, pending = "", ""
    for item in copilot.stream({"messages": [{"role": "user", "content": message}]}, cfg, context=REP,
                               stream_mode="messages", version="v2"):
        token, meta = item["data"]
        if meta["langgraph_node"] != "model" or not token.text:
            continue
        pending += token.text
        *sentences, pending = SENTENCE_END.split(pending)    # the finished sentences, and the start of the next
        for sentence in sentences:
            if checked(sentence)[1]:                           # a check failed: stop before the rep reads it
                yield SAFE_REPLY
                return
            shown += sentence + " "
            yield shown
    reply, failed = checked(copilot.get_state(cfg).values["messages"][-1].text)
    yield reply

In [8]:
import time

def watch(url, api_name, message):
    visitor, start, updates = Client(url, verbose=False), time.perf_counter(), []
    for update in visitor.submit(message, api_name=api_name):
        updates.append((round(time.perf_counter() - start, 1), update))
    return updates

page = gr.ChatInterface(respond_sentences)
page.launch(**LAUNCH)

# SCRIPTED MODEL · NO API CALL: a reply with the planted string in its second sentence.
copilot = create_agent(scripted(AIMessage(content="I read the newest note. " + PLANTED + ". Nothing else is needed.")),
                       tools=[], context_schema=RepContext, checkpointer=InMemorySaver())
print("scripted model:")
for seconds, update in watch(page.local_url, "/respond_sentences", FAIR):
    print(f"{seconds:5} s  {update}")

copilot = page_app.build_copilot()
updates = watch(page.local_url, "/respond_sentences", FAIR)
print(f"real copilot: {len(updates)} updates, first after {updates[0][0]} s, last after {updates[-1][0]} s")
for seconds, update in updates:
    print(f"{seconds:5} s  {update[-90:]}")
page.close()

scripted model:


  0.1 s  I read the newest note. 
  0.1 s  This reply failed a safety check and is not shown. Read the ticket in the ticket system.


real copilot: 1 updates, first after 4.1 s, last after 4.1 s
  4.1 s  album on November 2, 2013, providing both invoice numbers [ticket_16_overpayment_note.md].
Closing server running on port: 7860


**Result.** With the scripted model, the rep saw *I read the newest note.* and then the safe message; the planted string never reached the screen. With the real copilot the reply arrived in 1 update, after 4.1 s: it was one long sentence, so no sentence was finished before the end, and the rep saw nothing until the whole reply had been checked. Sentence checks show text early only when a reply has several sentences.

What a check on one sentence cannot see: anything split across sentences. A phone number written as *555. 0123 4567*
passes the check sentence by sentence and fails it as a whole:

In [9]:
reply = "Call 555. 0123 4567 today."
print("sentence by sentence:", [checked(sentence)[1] for sentence in SENTENCE_END.split(reply)])
print("whole reply:         ", checked(reply)[1])

sentence by sentence: [[], []]
whole reply:          ['no phone digits']


The check on the whole reply at the end still catches it, but by then the rep has read the pieces. Sentence checks
suit rules that hold for each sentence alone, such as a planted string.

In [10]:
if server is not None:
    server.should_exit = True